In [64]:
# %%
import pandas as pd
from pathlib import Path

DATA_IN  = Path("datos_organizados_escenas")  # carpeta raíz con Escena1/, Escena2/, …
DATA_OUT = Path("datos_sliding")               # carpeta de salida
ANCHOR_FILES = ["1.tsv", "2.tsv", "3.tsv", "4.tsv"]  # cambia si tienes más/menos
WIN_S   = 10   # tamaño de la ventana deslizante (segundos)
LIMIT   = None # máximo nº de segundos consecutivos a interpolar (None = sin límite)


In [65]:
# ## 2 ️⃣ Función de proceso

# %%
def process_file(path: Path, win_s: int = 10, interp_limit: int | None = None) -> pd.DataFrame:
    """Devuelve DF con epoch_s, distance_cm, rssi a 1 Hz."""

    # 1. Cargar TSV con cabecera
    df = pd.read_csv(path, sep="\t")

    # 2. Convertir timestamp ISO → datetime y fijar índice ord.
    df["timestamp_dt"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp_dt").sort_index()

    # 3. Ventana deslizante centrada
    roll = df.rolling(f"{win_s}s", center=True)
    out = roll.agg({
        "distance_cm": "mean",
        "rssi_dbm": "mean"
    })

    # 4. Resample a 1 Hz
    out = out.resample("1S").first()

    # 5. Generar rejilla completa de 1 Hz e interpolar / rellenar
    full_idx = pd.date_range(out.index.min(), out.index.max(), freq="1S")
    out = out.reindex(full_idx)

    out["distance_cm"] = out["distance_cm"].interpolate(
        method="time",
        limit=interp_limit,
        limit_direction="both"
    )
    out["rssi_dbm"] = out["rssi_dbm"].fillna(method="ffill").fillna(method="bfill")

    # 6. Epoch entero
    out = out.reset_index().rename(columns={"index": "ts"})
    out["epoch_s"] = (out["ts"].astype("int64") // 1_000_000_000).astype(int)

    return out[["epoch_s", "distance_cm", "rssi_dbm"]]

In [66]:
# %%
DATA_OUT.mkdir(parents=True, exist_ok=True)

for scene_dir in sorted(DATA_IN.glob("Escena*")):
    if not scene_dir.is_dir():
        continue
    out_scene = DATA_OUT / scene_dir.name
    out_scene.mkdir(parents=True, exist_ok=True)

    print(f"\n🔸 Procesando {scene_dir.name} …")
    for fname in ANCHOR_FILES:
        in_file = scene_dir / fname
        if not in_file.exists():
            print(f"  ⚠️  {fname} no existe, se omite")
            continue
        out_file = out_scene / fname
        df_out = process_file(in_file, win_s=WIN_S, interp_limit=LIMIT)
        df_out.to_csv(out_file, sep="\t", index=False, header=False)
        print(f"  ✔️  {fname} → {out_file.relative_to(DATA_OUT)}  ({len(df_out)} filas)")

print("\n✅  Conversión terminada. Archivos a 1 Hz en:", DATA_OUT)


🔸 Procesando Escena1 …
  ✔️  1.tsv → Escena1\1.tsv  (3164 filas)
  ✔️  2.tsv → Escena1\2.tsv  (3741 filas)
  ✔️  3.tsv → Escena1\3.tsv  (3157 filas)
  ✔️  4.tsv → Escena1\4.tsv  (3754 filas)

🔸 Procesando Escena2 …
  ✔️  1.tsv → Escena2\1.tsv  (3606 filas)
  ✔️  2.tsv → Escena2\2.tsv  (3605 filas)
  ✔️  3.tsv → Escena2\3.tsv  (3014 filas)
  ✔️  4.tsv → Escena2\4.tsv  (3582 filas)

🔸 Procesando Escena3 …
  ✔️  1.tsv → Escena3\1.tsv  (3560 filas)
  ✔️  2.tsv → Escena3\2.tsv  (3592 filas)
  ✔️  3.tsv → Escena3\3.tsv  (3592 filas)
  ✔️  4.tsv → Escena3\4.tsv  (3560 filas)

🔸 Procesando Escena4 …


C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:22: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  out = out.resample("1S").first()
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:25: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(out.index.min(), out.index.max(), freq="1S")
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:33: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  out["rssi_dbm"] = out["rssi_dbm"].fillna(method="ffill").fillna(method="bfill")
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:22: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  out = out.resample("1S").first()
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:25: Futu

  ✔️  1.tsv → Escena4\1.tsv  (3576 filas)
  ✔️  2.tsv → Escena4\2.tsv  (3577 filas)
  ✔️  3.tsv → Escena4\3.tsv  (2939 filas)
  ✔️  4.tsv → Escena4\4.tsv  (3577 filas)

🔸 Procesando Escena5 …
  ✔️  1.tsv → Escena5\1.tsv  (3617 filas)
  ✔️  2.tsv → Escena5\2.tsv  (3404 filas)
  ✔️  3.tsv → Escena5\3.tsv  (3087 filas)
  ✔️  4.tsv → Escena5\4.tsv  (3616 filas)

🔸 Procesando Escena6 …
  ✔️  1.tsv → Escena6\1.tsv  (3577 filas)
  ✔️  2.tsv → Escena6\2.tsv  (3201 filas)
  ✔️  3.tsv → Escena6\3.tsv  (3389 filas)
  ✔️  4.tsv → Escena6\4.tsv  (3708 filas)

✅  Conversión terminada. Archivos a 1 Hz en: datos_sliding


C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:22: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  out = out.resample("1S").first()
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:25: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(out.index.min(), out.index.max(), freq="1S")
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:33: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  out["rssi_dbm"] = out["rssi_dbm"].fillna(method="ffill").fillna(method="bfill")
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:22: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  out = out.resample("1S").first()
C:\Users\miguel\AppData\Local\Temp\ipykernel_11628\2804934086.py:25: Futu